# PHAN TICH XU HUONG TUYEN DUNG IT TAI VIET NAM
## 1. Phat bieu bai toan
Bai toan cua nhom la **phan tich du lieu nham khao sat xu huong tuyen dung IT tai Viet Nam** tu du lieu do nhom tu crawl tren cac nen tang tuyen dung. Trong tam cua notebook la mo ta cau truc thi truong tuyen dung, lam sach du lieu, ma hoa dac trung va rut ra insight tu cac bien ve **vai tro cong viec, dia diem, seniority va ky nang**.

O giai doan hien tai, nhom **khong chon mot bien muc tieu `Y` de du doan** lam trung tam, vi du lieu luong chi xuat hien tren mot tap con va khong du on dinh de xay dung mo hinh hoi quy dang tin cay. Do do, notebook di theo hai huong chinh:
- phan tich mo ta thi truong tuyen dung IT;
- chuan hoa du lieu va tao feature de phuc vu EDA, truc quan hoa va mo rong sang clustering neu can.

Tuy nhien, **nhanh phan tich luong van duoc giu lai** tren tap con co salary hop le de bo sung goc nhin ve thi truong lao dong IT, vi du so sanh muc luong theo `location_bucket`, `level`, `experience_band` va `job_title_group`.


### Dinh huong bai toan tren bo du lieu hien tai
- Huong phan tich phu hop nhat hien tai la **phan tich xu huong tuyen dung IT**, khong phai du doan luong.
- Luong duoc giu nhu **nhanh mo ta bo sung** tren tap con co salary hop le.
- Du lieu hien co phu hop de tra loi cac cau hoi nhu: nhom nghe nao tuyen nhieu, skill nao pho bien, thanh pho nao manh o nhom nghe nao, seniority gan voi nhung stack nao.
- Bo du lieu da duoc lam sach theo pipeline moi: hop nhat nhieu nguon, chuan hoa location/remote/level/company type, tach salary branch, trich skill va tao feature-ready dataset.
- Day la nen phu hop cho EDA hien tai va cung du tot de mo rong sang **clustering** o buoc sau.


## 2. Thiet lap du lieu va moi truong
Notebook nap truc tiep raw moi nhat trong `data/raw/` va cac file clean hien tai trong `data/clean-data/` de dam bao phan trinh bay khop voi pipeline dang dung.


In [ ]:
from pathlib import Path
import os
import warnings
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
os.environ.setdefault('MPLCONFIGDIR', str((Path('.') / '.mplconfig').resolve()))
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path('.')
RAW_DIR = BASE_DIR / 'data/raw'
CLEAN_DIR = BASE_DIR / 'data/clean-data'

RAW_ITVIEC = sorted(RAW_DIR.glob('itviec_jobs_*.csv'))[-1]
RAW_TOPCV = sorted(RAW_DIR.glob('topcv_jobs_*.csv'))[-1]
CLEAN_PATH = CLEAN_DIR / 'jobs_cleaned.csv'
FEATURE_PATH = CLEAN_DIR / 'jobs_features.csv'
SALARY_PATH = CLEAN_DIR / 'jobs_salary_subset.csv'


df_itviec = pd.read_csv(RAW_ITVIEC)
df_topcv = pd.read_csv(RAW_TOPCV)
df_raw_merged = pd.concat(
    [df_itviec.assign(source='itviec'), df_topcv.assign(source='topcv')],
    ignore_index=True,
    sort=False,
)
df_clean = pd.read_csv(CLEAN_PATH)
df_features = pd.read_csv(FEATURE_PATH)
df_salary = pd.read_csv(SALARY_PATH)

summary_loaded = pd.DataFrame(
    {
        'dataset': ['ITViec raw', 'TopCV raw', 'jobs_cleaned', 'jobs_features', 'jobs_salary_subset'],
        'rows': [len(df_itviec), len(df_topcv), len(df_clean), len(df_features), len(df_salary)],
        'columns': [df_itviec.shape[1], df_topcv.shape[1], df_clean.shape[1], df_features.shape[1], df_salary.shape[1]],
        'path': [str(RAW_ITVIEC), str(RAW_TOPCV), str(CLEAN_PATH), str(FEATURE_PATH), str(SALARY_PATH)],
    }
)
summary_loaded


## 3. Thu thap du lieu
Theo yeu cau hoc phan, du lieu duoc **tu thu thap** tu cac website tuyen dung thay vi dung dataset tai san. O snapshot hien tai, nhom dang su dung hai nguon raw chinh trong `data/raw/` la ITViec va TopCV.


In [ ]:
def parse_crawl_timestamp(path: Path) -> str:
    stem = path.stem
    parts = stem.split('_')
    if len(parts) >= 4:
        date_part, time_part = parts[-2], parts[-1]
        return pd.to_datetime(f'{date_part}{time_part}', format='%Y%m%d%H%M%S').strftime('%Y-%m-%d %H:%M:%S')
    return ''

collection_summary = pd.DataFrame(
    {
        'Nguon': ['ITViec', 'TopCV'],
        'File raw': [RAW_ITVIEC.name, RAW_TOPCV.name],
        'Script crawl': [
            'src/data_collection/itviec_crawler.py',
            'src/data_collection/topcv_crawler.py',
        ],
        'Cach thu thap': [
            'Selenium + BeautifulSoup, crawl URL roi di vao trang chi tiet',
            'requests + BeautifulSoup, crawl listing va trich noi dung HTML',
        ],
        'So mau raw': [len(df_itviec), len(df_topcv)],
        'So bien raw': [df_itviec.shape[1], df_topcv.shape[1]],
        'Ngay crawl tu ten file': [parse_crawl_timestamp(RAW_ITVIEC), parse_crawl_timestamp(RAW_TOPCV)],
    }
)
collection_summary.loc[len(collection_summary)] = [
    'Tong cong', '-', '-', 'Du lieu do nhom tu crawl tu 2 nguon', len(df_raw_merged), df_raw_merged.shape[1], '-'
]
collection_summary


### Schema du lieu raw
Hai nguon raw co phan loi kha giong nhau, nhung van co khac biet schema can xu ly khi merge:
- Cot loi chung: `job_title`, `tech_stack`, `experience_years`, `location`, `company_name`, `remote_option`, `salary_min`, `salary_max`, `currency`, `posted_date`, `job_description`.
- ITViec co them `company_type`, `company_industry`.
- TopCV co them `deadline`.

Vi vay pipeline clean hien tai di theo huong **union schema**, giu lai cac cot chi co o mot nguon va chi chuan hoa/chuyen doi o buoc cleaning sau do.


In [ ]:
print('Cac cot cua ITViec:', list(df_itviec.columns))
print('Cac cot cua TopCV :', list(df_topcv.columns))

print('
Mau du lieu tho ITViec:')
display(df_itviec.head(3))

print('Mau du lieu tho TopCV:')
display(df_topcv.head(3))


## 4. Mo ta du lieu ban dau
Truoc khi phan tich xu huong, can kiem tra quy mo mau, muc do thieu du lieu va do lech giua cac nguon. Day la buoc nen de danh gia du lieu co du sach va du manh cho EDA hay khong.


In [ ]:
raw_overview = pd.DataFrame(
    {
        'Chi so': [
            'So mau raw sau khi gop',
            'So nguon du lieu',
            'So bien raw',
            'So mau clean',
            'So mau feature-ready',
            'So mau co salary_avg',
            'Ty le mau co salary_avg',
        ],
        'Gia tri': [
            len(df_raw_merged),
            df_raw_merged['source'].nunique(),
            df_raw_merged.shape[1],
            len(df_clean),
            len(df_features),
            int(df_clean['has_salary'].sum()),
            round(df_clean['has_salary'].mean(), 4),
        ],
    }
)
raw_overview


In [ ]:
raw_missing = df_raw_merged.isna().mean().sort_values(ascending=False).head(12).to_frame('missing_ratio_raw')
clean_missing = df_clean.isna().mean().sort_values(ascending=False).head(12).to_frame('missing_ratio_clean')

display(raw_missing)
display(clean_missing)

## 5. Thong ke mo ta va truc quan hoa don bien
O buoc nay, notebook mo ta nhanh thi truong tuyen dung IT thong qua cac bien quan trong nhat cho bai toan xu huong: `job_title_group`, `level`, `location_bucket`, `company_type`, `remote_option`, `source` va `skills_extracted`. Nhanh salary chi dung tren `df_salary`.


In [ ]:
source_counts = df_clean['source'].value_counts()
top_locations = df_clean['location_bucket'].fillna('Thieu').value_counts()
top_levels = df_clean['level'].fillna('Thieu').value_counts()
top_roles = df_clean['job_title_group'].fillna('Thieu').value_counts().head(15)
top_company_types = df_clean['company_type'].fillna('Thieu').value_counts()
salary_summary = df_salary[['salary_min', 'salary_max', 'salary_avg']].describe().T

display(source_counts.to_frame('So tin'))
display(top_locations.to_frame('So tin'))
display(top_levels.to_frame('So tin'))
display(top_roles.to_frame('So tin'))
display(top_company_types.to_frame('So tin'))
display(salary_summary)


In [ ]:
skill_freq = (
    df_features['skills_extracted']
    .fillna('')
    .str.split(', ')
    .explode()
)
skill_freq = skill_freq[skill_freq.ne('')].value_counts().head(15)

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

sns.countplot(data=df_clean, x='source', order=source_counts.index, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('So luong tin theo nguon')
axes[0, 0].set_xlabel('source')
axes[0, 0].set_ylabel('So tin')

sns.countplot(data=df_clean, y='job_title_group', order=top_roles.index, ax=axes[0, 1], color='#4c78a8')
axes[0, 1].set_title('Nhom vi tri duoc tuyen nhieu nhat')
axes[0, 1].set_xlabel('So tin')
axes[0, 1].set_ylabel('job_title_group')

sns.countplot(data=df_clean, y='location_bucket', order=top_locations.index, ax=axes[0, 2], color='#f58518')
axes[0, 2].set_title('Phan bo location bucket sau cleaning')
axes[0, 2].set_xlabel('So tin')
axes[0, 2].set_ylabel('location_bucket')

sns.barplot(x=skill_freq.values, y=skill_freq.index, ax=axes[1, 0], color='#54a24b')
axes[1, 0].set_title('Top 15 skill xuat hien nhieu nhat')
axes[1, 0].set_xlabel('So lan xuat hien')
axes[1, 0].set_ylabel('Skill')

sns.histplot(df_salary['salary_avg'] / 1_000_000, bins=25, kde=True, ax=axes[1, 1], color='#b279a2')
axes[1, 1].set_title('Phan bo salary_avg tren tap con co luong')
axes[1, 1].set_xlabel('salary_avg (trieu VND/thang)')
axes[1, 1].set_ylabel('So tin')

salary_level_order = [lvl for lvl in ['Intern/Fresher', 'Junior', 'Middle', 'Senior', 'Lead', 'Manager', 'Director/C-level'] if lvl in df_salary['level'].dropna().unique()]
sns.boxplot(data=df_salary, x='level', y=df_salary['salary_avg'] / 1_000_000, order=salary_level_order, ax=axes[1, 2], color='#ff9da6')
axes[1, 2].set_title('Muc luong theo level (tap con co luong)')
axes[1, 2].set_xlabel('level')
axes[1, 2].set_ylabel('salary_avg (trieu VND)')
axes[1, 2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


### Nhan xet nhanh tu thong ke don bien
- Sau cleaning, `location_bucket` da duoc gom ve dung cac hub quan trong cho phan tich thi truong.
- `job_title_group` phan anh kha ro cau truc nghe nghiep: cac nhom backend, full-stack, mobile va front-end noi bat hon han.
- `skills_extracted` va cac top skill la loi cua phan phan tich xu huong cong nghe.
- Tap con co luong van du de minh hoa phan bo salary theo `level` hoac `job_title_group`, nhung khong duoc dung nhu truc chinh cua bai toan.


## 6. Lam sach va chuan hoa du lieu
Pipeline cleaning hien tai bam dung workflow da chot va duoc thuc hien theo thu tu sau:

1. **Hop nhat raw theo schema chung**
- Doc hai file raw moi nhat tu ITViec va TopCV.
- Giu lai toan bo cot cua hai nguon theo huong union schema, khong loai som cac cot chi xuat hien o mot nguon nhu `company_industry` hay `deadline`.
- Bo sung bien `source` de theo doi nguon du lieu va `crawl_date` de giu lai thong tin snapshot.

2. **Chuan hoa text va kieu du lieu co ban**
- Cac cot text nhu `job_title`, `company_name`, `location`, `remote_option`, `tech_stack`, `company_type`, `company_industry`, `job_description` duoc trim, chuan hoa Unicode NFC, rut gon khoang trang thua va dua ve dang text nhat quan.
- Cac cot ngay nhu `posted_date`, `deadline`, `crawl_date` duoc ep sang `datetime`.
- Cac cot so nhu `salary_min`, `salary_max`, `experience_years` duoc parse ve numeric de phuc vu thong ke va tao feature.

3. **Lam sach va chuan hoa dia diem**
- Tach va chuan hoa truong `location` thanh `location_list` de xu ly cac tin co nhieu dia diem.
- Xac dinh `primary_city` la thanh pho chinh dai dien cho tin tuyen dung.
- Tao bien `is_multi_location` de danh dau cac tin co tu hai dia diem tro len.
- Gom dia diem ve 6 nhom thi truong thong nhat qua bien `location_bucket`: `Ha Noi`, `Ho Chi Minh`, `Da Nang`, `Can Tho`, `Hai Phong`, `Others`.

4. **Chuan hoa hinh thuc lam viec**
- Giu lai gia tri goc qua `remote_option_raw`.
- Chuan hoa `remote_option` ve 3 nhan chinh: `onsite`, `hybrid`, `remote`.
- Neu khong du thong tin thi de trong thay vi suy dien tu `location`.

5. **Chuan hoa kinh nghiem va seniority**
- Parse `experience_years` tu nhieu dang khac nhau nhu `2`, `2 years`, `5+ years`, `3-5 years`.
- Tao `experience_band` theo cac nhom `<1`, `1-2`, `3-4`, `5-8`, `>8` de phu hop voi huong phan tich theo bao cao ITViec.
- Chuan hoa `level` ve cac nhom `Intern/Fresher`, `Junior`, `Middle`, `Senior`, `Lead`, `Manager`, `Director/C-level`, `Unknown`.
- Neu du lieu goc khong co `level`, pipeline moi suy ra tu `job_title` hoac `experience_years`, dong thoi gan co `level_inferred`.

6. **Chuan hoa taxonomy vi tri cong viec**
- Tu `job_title`, `tech_stack` va `job_description`, cac title duoc map ve nhom `job_title_group` de giam nhieu title le.
- Taxonomy nay gom cac nhom nhu back-end, front-end, full-stack, mobile, data, tester, business analyst, project manager, security/devops/cloud, ERP, IT support/helpdesk va `Other IT Roles`.
- Muc tieu la dua du lieu ve muc nhom vai tro co the so sanh truc tiep trong EDA va visualization.

7. **Xu ly salary thanh mot nhanh rieng**
- Giu nguyen gia tri goc qua `salary_min_original`, `salary_max_original`, `currency_original`.
- Chuan hoa don vi tien te ve `VND` khi co du thong tin quy doi va tao `salary_avg`.
- Tao cac bien ho tro phan tich luong gom `has_salary`, `salary_currency`, `salary_band`.
- Khong impute salary cho toan bo dataset; chi tao tap con `jobs_salary_subset.csv` cho nhung dong co salary hop le.

8. **Chuan hoa company type theo muc do tin cay**
- Giu lai gia tri goc qua `company_type_raw`.
- Map `company_type` ve cac nhom `Product`, `Outsource`, `Service/Consulting`, `Non-IT`, `Startup` khi co tin hieu du manh.
- Neu gia tri duoc suy ra thay vi den truc tiep tu raw thi danh dau bang `company_type_inferred = 1`.
- Truong hop khong du bang chung thi de trong, tranh suy dien qua tay.

9. **Khu trung lap theo khoa nghiep vu**
- Tao cac khoa trung gian nhu `title_key`, `company_key`, `dedup_key`, `fuzzy_group_key` sau khi chuan hoa title va ten cong ty.
- Loai bo duplicate exact-match dua tren `dedup_key`.
- Thuc hien them mot vong fuzzy dedup de bat cac tin rat giong nhau nhung khac mot it ve format title hoac company name.

10. **Tao feature sau cleaning**
- Trich `skills_extracted` tu `job_title`, `tech_stack`, `job_description`.
- Tao them `skills_count`, `main_language`, `main_framework`, `skill_domain`.
- Sinh cac bien indicator nhu `has_python`, `has_js_ts`, `has_sql`, `has_cloud`, `has_data`, `has_devops`, `has_ai`, `has_backend`, `has_frontend`, `has_testing`, `has_mobile`.

11. **Xuat bo du lieu sau cung**
- `jobs_cleaned.csv`: bo du lieu da duoc lam sach de phuc vu EDA tong quat.
- `jobs_features.csv`: bo du lieu da co them cac feature skill va indicator de truc quan hoa da bien, embedding va clustering.
- `jobs_salary_subset.csv`: tap con salary hop le de thong ke luong.


### Minh hoa buoc 1: hop nhat raw theo schema chung
Cell duoi day cho thay quy mo tung nguon, so cot cua tung file raw va nhung cot chi xuat hien o mot nguon truoc khi union schema.


In [ ]:
schema_compare = pd.DataFrame(
    {
        'dataset': ['ITViec raw', 'TopCV raw', 'Merged raw'],
        'rows': [len(df_itviec), len(df_topcv), len(df_raw_merged)],
        'columns': [df_itviec.shape[1], df_topcv.shape[1], df_raw_merged.shape[1]],
    }
)

unique_itviec_cols = sorted(set(df_itviec.columns) - set(df_topcv.columns))
unique_topcv_cols = sorted(set(df_topcv.columns) - set(df_itviec.columns))

display(schema_compare)
display(pd.DataFrame({'ITViec_only_columns': pd.Series(unique_itviec_cols), 'TopCV_only_columns': pd.Series(unique_topcv_cols)}))


### Minh hoa buoc 2: chuan hoa text va kieu du lieu
Buoc nay dua cac cot text, ngay thang va numeric ve dang nhat quan de cac buoc phan tich sau khong bi loi do kieu du lieu.


In [ ]:
dtype_compare = pd.DataFrame(
    {
        'raw_dtype': df_raw_merged[['posted_date', 'salary_min', 'salary_max', 'experience_years']].dtypes.astype(str),
        'clean_dtype': df_clean[['posted_date', 'salary_min', 'salary_max', 'experience_years']].dtypes.astype(str),
    }
)

text_preview = df_clean[['job_title', 'company_name', 'location', 'remote_option_raw']].head(8)

display(dtype_compare)
display(text_preview)


### Minh hoa buoc 3 va 4: chuan hoa dia diem va hinh thuc lam viec
Tu `location` goc, pipeline xac dinh `primary_city`, gom ve `location_bucket`, dong thoi chuan hoa `remote_option` ve 3 nhan thong nhat.


In [ ]:
location_preview = df_clean[
    ['location', 'primary_city', 'location_bucket', 'is_multi_location', 'remote_option_raw', 'remote_option']
].drop_duplicates().head(12)

remote_compare = pd.concat(
    [
        df_raw_merged['remote_option'].fillna('Missing').value_counts().rename('raw_count'),
        df_clean['remote_option'].fillna('Missing').value_counts().rename('clean_count'),
    ],
    axis=1,
).fillna(0).astype(int)

display(location_preview)
display(remote_compare)


### Minh hoa buoc 5: chuan hoa kinh nghiem va level
Kinh nghiem duoc parse ve dang so, sau do nhom lai thanh `experience_band`. Neu raw khong co `level`, pipeline suy ra co kiem soat va danh dau bang `level_inferred`.


In [ ]:
experience_level_preview = df_clean[
    ['job_title', 'experience_years', 'experience_band', 'level', 'level_inferred']
].sample(min(12, len(df_clean)), random_state=42)

experience_band_counts = df_clean['experience_band'].fillna('Missing').value_counts().to_frame('count')

display(experience_level_preview)
display(experience_band_counts)


### Minh hoa buoc 6: chuan hoa taxonomy vi tri cong viec
Nhieu title chi tiet duoc map ve `job_title_group` de giam nhieu va dua du lieu ve muc role co the so sanh truc tiep.


In [ ]:
role_mapping_preview = df_clean[['job_title', 'job_title_group']].drop_duplicates().sample(min(15, df_clean[['job_title', 'job_title_group']].drop_duplicates().shape[0]), random_state=42)
role_distribution = df_clean['job_title_group'].value_counts().head(12).to_frame('count')

display(role_mapping_preview)
display(role_distribution)


### Minh hoa buoc 7 va 8: xu ly salary va company type
Salary duoc tach thanh nhanh rieng va company type chi duoc suy ra khi du bang chung. Cac cot duoi day cho thay ket qua sau chuan hoa.


In [ ]:
salary_company_preview = df_clean[
    [
        'job_title', 'salary_min_original', 'salary_max_original', 'currency_original',
        'salary_avg', 'salary_band', 'has_salary', 'company_type', 'company_type_inferred'
    ]
].head(12)

salary_company_summary = pd.DataFrame(
    {
        'metric': ['salary_rows', 'salary_coverage', 'inferred_company_type_rows'],
        'value': [int(df_clean['has_salary'].sum()), round(df_clean['has_salary'].mean(), 4), int(df_clean['company_type_inferred'].sum())],
    }
)

display(salary_company_preview)
display(salary_company_summary)


### Minh hoa buoc 9 den 11: dedup, feature sau cleaning va tap dau ra cuoi cung
Buoc cuoi cung loai trung lap, giu lai bo clean on dinh, tao feature phuc vu EDA va tach tap salary hop le.


In [ ]:
standardized_preview = df_clean[
    [
        'job_id', 'job_title', 'location_bucket', 'remote_option',
        'experience_band', 'level', 'job_title_group', 'company_type', 'has_salary'
    ]
].head(12)

cleaning_summary = pd.DataFrame(
    {
        'Chi so': [
            'So mau raw sau khi gop',
            'So mau sau cleaning',
            'So mau bi loai sau cleaning + dedup',
            'So job_id trung lap trong clean',
            'So location bucket sau chuan hoa',
            'So nhan remote_option sau chuan hoa',
            'So nhom vi tri sau chuan hoa',
            'So dong co salary hop le',
            'So dong trong jobs_features',
            'So dong trong jobs_salary_subset',
        ],
        'Gia tri': [
            len(df_raw_merged),
            len(df_clean),
            len(df_raw_merged) - len(df_clean),
            int(df_clean['job_id'].duplicated().sum()),
            int(df_clean['location_bucket'].nunique(dropna=True)),
            int(df_clean['remote_option'].nunique(dropna=True)),
            int(df_clean['job_title_group'].nunique(dropna=True)),
            int(df_clean['has_salary'].sum()),
            len(df_features),
            len(df_salary),
        ],
    }
)

display(standardized_preview)
cleaning_summary


### Y nghia cua buoc cleaning
- `location_bucket` giup dua du lieu ve dung cac hub thi truong de tranh phan manh gia tao trong phan tich dia ly.
- `job_title_group` giup gom nhieu title chi tiet thanh cac nhom nghe co the so sanh truc tiep.
- `company_type` duoc infer co kiem soat va luon giu co `company_type_inferred` de phan biet voi du lieu goc.
- `jobs_salary_subset.csv` tach rieng phan luong de tranh lam nhieu EDA chinh.
- Sau cleaning, bo du lieu da du nhat quan de dua truc tiep vao EDA, visualization va cac buoc feature engineering tiep theo.


## 7. Ma hoa du lieu va xu ly ngon ngu tu nhien
Text tuyen dung duoc chuan hoa de trich skill va tao feature co the dung truc tiep cho EDA va embedding. Day la phan noi giua cleaning voi feature engineering.


In [ ]:
feature_preview = df_features[
    [
        'job_title', 'job_title_group', 'location_bucket', 'level', 'company_type',
        'skills_extracted', 'skills_count', 'main_language', 'main_framework', 'skill_domain',
        'has_python', 'has_js_ts', 'has_sql', 'has_cloud', 'has_data', 'has_devops',
        'has_backend', 'has_frontend'
    ]
].head(10)

display(feature_preview)

indicator_cols = [
    'has_ai', 'has_python', 'has_java', 'has_js_ts', 'has_sql',
    'has_cloud', 'has_data', 'has_devops', 'has_mobile', 'has_testing',
    'has_backend', 'has_frontend'
]
indicator_rate = df_features[indicator_cols].mean().sort_values(ascending=False).to_frame('Ty le xuat hien')
indicator_rate


## 8. Xay dung va lua chon dac trung
Voi bai toan phan tich xu huong tuyen dung, tap feature huu ich nhat khong xoay quanh salary ma xoay quanh role, seniority, city bucket va skill. Salary chi nen dung o nhanh thong ke mo ta tren tap con co luong.


In [ ]:
feature_catalog = pd.DataFrame(
    [
        ['job_title_group', 'category', 'Nhom vai tro cong viec', 'Rat quan trong'],
        ['level', 'ordinal/category', 'Cap bac tuyen dung', 'Rat quan trong'],
        ['experience_years', 'numeric', 'So nam kinh nghiem', 'Huu ich'],
        ['experience_band', 'category', 'Nhom kinh nghiem theo bao cao ITviec', 'Huu ich'],
        ['location_bucket', 'category', 'Hub dia ly chuan hoa', 'Rat quan trong'],
        ['company_type', 'category', 'Loai hinh doanh nghiep', 'Huu ich'],
        ['remote_option', 'category', 'Hinh thuc lam viec', 'Huu ich'],
        ['skills_extracted', 'text/list', 'Danh sach skill da trich', 'Rat quan trong'],
        ['main_language', 'category', 'Ngon ngu chinh noi bat', 'Huu ich'],
        ['main_framework', 'category', 'Framework chinh noi bat', 'Huu ich'],
        ['has_python, has_js_ts, has_sql, ...', 'binary', 'Nhom chi bao skill', 'Rat quan trong'],
    ],
    columns=['Dac trung', 'Kieu du lieu', 'Y nghia', 'Danh gia'],
)
feature_catalog


### Tap dac trung huu ich de xuat
Cac dac trung nen uu tien khi viet EDA va ket luan:
- `job_title_group`, `level`, `experience_band`, `location_bucket`, `company_type`, `remote_option`
- `skills_extracted`, `skills_count`, `main_language`, `main_framework`, `skill_domain`
- cac bien nhi phan skill nhu `has_python`, `has_js_ts`, `has_sql`, `has_cloud`, `has_data`, `has_devops`, `has_ai`, `has_backend`, `has_frontend`

Day la tap feature vua giau y nghia nghiep vu, vua du manh de tiep tuc truc quan hoa da bien hoac lam embedding cho clustering.


## 9. Truc quan hoa moi quan he da bien
Phan nay truc tiep tra loi cac cau hoi trung tam cua de tai: thanh pho nao manh o nhom nghe nao, level nao gan voi nhom skill nao, va cac combo skill nao xuat hien cung nhau nhieu nhat.


In [ ]:
role_location = pd.crosstab(df_clean['job_title_group'], df_clean['location_bucket'])
role_location_top = role_location.loc[role_location.sum(axis=1).sort_values(ascending=False).head(10).index]
role_location_top = role_location_top[role_location_top.sum(axis=0).sort_values(ascending=False).index]

level_skill = df_features.groupby('level')[indicator_cols].mean().loc[
    [lvl for lvl in ['Intern/Fresher', 'Junior', 'Middle', 'Senior', 'Lead', 'Manager', 'Director/C-level', 'Unknown'] if lvl in df_features['level'].dropna().unique()]
]

top_salary_roles = df_salary['job_title_group'].value_counts().head(8).index

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.heatmap(role_location_top, cmap='YlGnBu', annot=True, fmt='g', ax=axes[0, 0])
axes[0, 0].set_title('Heatmap so tin theo job_title_group va location_bucket')
axes[0, 0].set_xlabel('location_bucket')
axes[0, 0].set_ylabel('job_title_group')

sns.heatmap(level_skill, cmap='OrRd', annot=True, fmt='.2f', ax=axes[0, 1])
axes[0, 1].set_title('Ty le xuat hien skill indicator theo level')
axes[0, 1].set_xlabel('Skill indicator')
axes[0, 1].set_ylabel('level')

sns.countplot(
    data=df_clean,
    y='job_title_group',
    hue='source',
    order=df_clean['job_title_group'].value_counts().head(10).index,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Top nhom nghe theo tung nguon')
axes[1, 0].set_xlabel('So tin')
axes[1, 0].set_ylabel('job_title_group')

sns.boxplot(
    data=df_salary[df_salary['job_title_group'].isin(top_salary_roles)],
    x='job_title_group',
    y=df_salary[df_salary['job_title_group'].isin(top_salary_roles)]['salary_avg'] / 1_000_000,
    ax=axes[1, 1],
    color='#72b7b2'
)
axes[1, 1].set_title('Muc luong theo nhom nghe (tap con co luong)')
axes[1, 1].set_xlabel('job_title_group')
axes[1, 1].set_ylabel('salary_avg (trieu VND)')
axes[1, 1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
skill_lists = (
    df_features['skills_extracted']
    .fillna('')
    .str.split(', ')
    .apply(lambda items: [item for item in items if item])
)

pair_counter = Counter()
for items in skill_lists:
    for pair in combinations(sorted(set(items)), 2):
        pair_counter[pair] += 1

top_pairs = pd.DataFrame(pair_counter.most_common(15), columns=['pair', 'count'])
if not top_pairs.empty:
    top_pairs['pair'] = top_pairs['pair'].apply(lambda value: ' + '.join(value))
    plt.figure(figsize=(12, 6))
    sns.barplot(data=top_pairs, x='count', y='pair', color='#4c78a8')
    plt.title('Top cap skill dong xuat hien nhieu nhat')
    plt.xlabel('So lan dong xuat hien')
    plt.ylabel('Cap skill')
    plt.tight_layout()
    plt.show()

top_pairs.head(10)


### Dien giai tu truc quan hoa da bien
- Heatmap `job_title_group` theo `location_bucket` cho thay hub nao manh o nhom nghe nao.
- Ma tran skill theo `level` giup tra loi cau hoi seniority nao gan voi nhung nhom cong nghe nao.
- Top cap skill dong xuat hien la bang chung truc tiep cho cac combo cong nghe pho bien tren thi truong.
- Tren tap con co luong, boxplot theo `job_title_group` cho them mot lop dien giai ve phan khuc nhan luc.


## 10. Truc quan hoa khong gian du lieu nhieu chieu
Du bai toan hien tai la phan tich xu huong, ta van co the nhung cac dac trung da ma hoa xuong khong gian 2 chieu de quan sat cau truc nghe nghiep tong the. Day la buoc goi mo neu sau nay muon mo rong sang clustering.


In [ ]:
embedding_features = [
    'location_bucket', 'company_type', 'level', 'remote_option', 'job_title_group',
    'experience_years', 'skills_count', 'has_ai', 'has_python', 'has_java',
    'has_js_ts', 'has_sql', 'has_cloud', 'has_data', 'has_devops',
    'has_mobile', 'has_testing', 'has_backend', 'has_frontend'
]

tsne_sample = df_features[embedding_features].copy()
if len(tsne_sample) > 400:
    tsne_sample = tsne_sample.sample(400, random_state=42)

categorical_cols = ['location_bucket', 'company_type', 'level', 'remote_option', 'job_title_group']
numeric_cols = [col for col in embedding_features if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore')),
                ]
            ),
            categorical_cols,
        ),
        (
            'num',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]
            ),
            numeric_cols,
        ),
    ]
)

X_emb = preprocessor.fit_transform(tsne_sample)
X_emb = X_emb.toarray() if hasattr(X_emb, 'toarray') else X_emb

coords = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto', perplexity=30).fit_transform(X_emb)
plot_df = df_features.loc[tsne_sample.index, ['job_title_group', 'level']].copy()
plot_df['x'] = coords[:, 0]
plot_df['y'] = coords[:, 1]

plt.figure(figsize=(12, 8))
sns.scatterplot(data=plot_df, x='x', y='y', hue='job_title_group', style='level', s=80, alpha=0.8)
plt.title('t-SNE tren khong gian feature tuyen dung IT')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


### Nhan xet ve khong gian nhieu chieu
- Neu cac diem co xu huong tu lai theo `job_title_group` hoac `level`, dieu do cho thay du lieu co cau truc nghe nghiep tuong doi ro.
- Day la tin hieu tich cuc neu sau nay nhom muon di tiep sang bai toan phan cum.
- Trong notebook hien tai, t-SNE chu yeu dong vai tro minh hoa cho cau truc nhieu chieu cua du lieu da duoc clean va feature hoa.


## 11. Danh gia tinh kha thi cua bai toan
Phan cuoi cung tong hop xem du lieu hien tai co that su phu hop cho huong phan tich xu huong tuyen dung IT hay khong.


In [ ]:
feasibility_table = pd.DataFrame(
    [
        ['Tu thu thap du lieu', 'Dat', 'Co raw crawl thuc te tu ITViec va TopCV'],
        ['So luong mau', 'Dat', f'Raw hien tai co {len(df_raw_merged)} mau, clean con {len(df_clean)} mau'],
        ['Co dac trung phu hop cho xu huong', 'Dat', 'Co job_title_group, level, location_bucket, company_type, remote_option, skill'],
        ['Co du lieu text de lam NLP', 'Dat', 'Co tech_stack va job_description de trich skill'],
        ['Co nhanh salary mo ta bo sung', 'Dat', f'Tap con salary hien co {len(df_salary)} mau hop le'],
        ['Co the mo rong sang clustering', 'Kha thi', 'Da co bo feature category/binary/numeric tuong doi day du'],
        ['Rui ro du lieu con lai', 'Co nhung kiem soat duoc', 'Mot so dong level Unknown, company_type infer co co, salary khong day du'],
    ],
    columns=['Tieu chi', 'Danh gia', 'Giai thich'],
)
feasibility_table


### Ket luan ve tinh kha thi
**Huong phan tich xu huong tuyen dung IT la phu hop va kha thi hon han so voi viec lay salary lam target chinh o thoi diem hien tai.**

Ly do:
- Du lieu raw da vuot nguong toi thieu ve so mau va do nhom tu crawl.
- Pipeline clean hien tai du manh de kiem soat duplicate, location, remote option, role taxonomy, company type va salary branch.
- Du lieu text da duoc chuyen thanh feature co the thong ke, truc quan hoa va dung cho NLP muc co ban.
- Du salary bi thieu o nhieu dong, tap con co luong van du de minh hoa them cho thi truong ma khong lam lech huong bai toan chinh.


## 12. Ket luan
Notebook da chuyen trong tam thanh cong tu y tuong du doan luong sang **phan tich xu huong tuyen dung IT tai Viet Nam**. Day la huong phu hop hon voi chat luong du lieu hien co va cung bam sat thuc te hon.

Ket luan chinh:
- Bo du lieu hien tai phu hop de khao sat xu huong tuyen dung theo **nhom nghe, city bucket, seniority level, loai hinh doanh nghiep va ky nang**.
- Cac cau hoi nhu nhom vi tri nao tuyen nhieu nhat, skill nao pho bien nhat, skill nao thuong di cung nhau, hub nao manh o nhom nghe nao deu co the tra loi bang du lieu hien co.
- `jobs_cleaned.csv`, `jobs_features.csv` va `jobs_salary_subset.csv` tao thanh bo du lieu du tot cho EDA hien tai va cac buoc mo rong tiep theo.
- Bo du lieu sau cleaning da du nhat quan de tiep tuc truc quan hoa, phan tich da bien va mo rong sang clustering neu can.


## 13. Tai lieu tham khao
- [description.ipynb](description.ipynb): notebook mo ta bai toan va EDA tong hop.
- [data/raw](data/raw): du lieu raw do nhom crawl tu ITViec va TopCV.
- [jobs_cleaned.csv](data/clean-data/jobs_cleaned.csv): du lieu da lam sach de phuc vu EDA tong quat.
- [jobs_features.csv](data/clean-data/jobs_features.csv): bo feature da trich skill va ma hoa phuc vu phan tich sau.
- [jobs_salary_subset.csv](data/clean-data/jobs_salary_subset.csv): tap con gom cac mau co salary hop le de thong ke luong.
- [ITviec_Salary_Report_2025_2026_EN.pdf](d:/ITviec_Salary_Report_2025_2026_EN.pdf): tai lieu tham chieu ve xu huong salary va taxonomy vi tri.
- Tai lieu tham khao ve feature engineering: <https://phamdinhkhanh.github.io/2019/01/07/Ky_thuat_feature_engineering.html>
- Tai lieu tham khao ve t-SNE: <https://www.datacamp.com/tutorial/introduction-t-sne>
